In [1]:
import sys, os, math, time, csv
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    !nvidia-smi 


sys.path.insert(0, ".")
import shooter  # registers Shooter-v0

print("Environment registered: Shooter-v0")

Using device: cuda
Sat May  9 13:55:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.120                Driver Version: 550.120        CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     Off |   00000000:84:00.0 Off |                  N/A |
| 24%   40C    P8             26W /  250W |     226MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------

In [2]:
# ============================================================
# Cell 2: LowLevelLeadAimAgent (expert dataset)
# ============================================================
AI_SPEED     = 0.35
BULLET_SPEED = 2.5
TREE_R       = 2.5

def wrap_angle(angle: float) -> float:
    return (angle + math.pi) % (2.0 * math.pi) - math.pi

def extract_targets(obs):
    targets = []
    for slot in range(15):
        base = 6 + slot * 8
        if obs[base+6] > 0.5 and obs[base+7] > 0.5:
            x = float(obs[base]   * 100.0)
            z = float(obs[base+1] * 100.0)
            d = float(obs[base+2] * 141.0)
            y = float(obs[base+5] * 20.0)
            targets.append({"x":x, "y":y, "z":z, "dist":max(d,0.5)})
    return targets

def compute_intercept_aim(target, lead=1.30):
    x,y,z = target["x"], target["y"], target["z"]
    dist_xz = max(math.sqrt(x*x+z*z), 0.5)
    px,py,pz = x, y-2.3, z
    vx = -x/dist_xz * AI_SPEED
    vy = 0.0
    vz = -z/dist_xz * AI_SPEED
    a = vx*vx+vy*vy+vz*vz - BULLET_SPEED*BULLET_SPEED
    b = 2.0*(px*vx+py*vy+pz*vz)
    c = px*px+py*py+pz*pz
    disc = b*b-4*a*c
    if disc >= 0 and abs(a)>1e-8:
        root = math.sqrt(disc)
        t1 = (-b-root)/(2*a)
        t2 = (-b+root)/(2*a)
        valid = [t for t in (t1,t2) if 0<t<120]
        T = min(valid) if valid else dist_xz/BULLET_SPEED
    else:
        T = dist_xz/BULLET_SPEED
    T = max(0.0, min(T*lead, 120.0))
    aim_x = px + vx*T
    aim_y = py + vy*T
    aim_z = pz + vz*T
    req_yaw = math.atan2(aim_z, aim_x)
    req_pitch = math.atan2(aim_y, max(math.sqrt(aim_x**2+aim_z**2),0.5))
    return req_yaw, req_pitch

class LowLevelLeadAimAgent:
    def __init__(self):
        self.last_fire_tick = -9999
        self.fire_cooldown = 8
        self.max_fire_dist = 85.0
        self.min_fire_dist = 8.0
        self.max_bullets   = 1
        self.lead_scale    = 1.30

    def fire_tolerances(self, dist):
        yaw_tol   = max(0.065, min(0.16, 3.0/max(dist,1)))
        pitch_tol = max(0.045, min(0.14, 2.5/max(dist,1)))
        return yaw_tol, pitch_tol

    def choose_nearest(self, obs):
        targets = extract_targets(obs)
        if not targets: return None
        practical = [t for t in targets if t["dist"]>=self.min_fire_dist]
        return min(practical if practical else targets, key=lambda t: t["dist"])

    def action(self, obs, info):
        target = self.choose_nearest(obs)
        if target is None:
            return 0  # do_nothing
        cur_yaw   = float(obs[3]*math.pi)
        cur_pitch = float(obs[4]*0.9)
        req_yaw, req_pitch = compute_intercept_aim(target, self.lead_scale)
        yaw_err = wrap_angle(req_yaw - cur_yaw)
        pitch_err = req_pitch - cur_pitch
        dist = float(target["dist"])
        yt, pt = self.fire_tolerances(dist)
        aligned = abs(yaw_err)<=yt and abs(pitch_err)<=pt
        tick = int(info.get("tick",0))
        bullets = int(info.get("bullets_in_flight",0))
        cd = (tick - self.last_fire_tick) >= self.fire_cooldown
        dist_ok = self.min_fire_dist <= dist <= self.max_fire_dist
        bullet_ok = bullets <= self.max_bullets
        if aligned and cd and dist_ok and bullet_ok:
            self.last_fire_tick = tick
            return 1  # fire
        
        if abs(yaw_err) > 0.04:
            return 2 if yaw_err > 0 else 3
        if abs(pitch_err) > 0.014:
            return 4 if pitch_err > 0 else 5
        return 0  # do_nothing


In [3]:
# ============================================================
# Cell 3: Generate dataset from expert
# ============================================================
def generate_dataset(num_episodes=300, seed_offset=0):
    env = gym.make("Shooter-v0", render_mode=None)
    data_obs = []
    data_act = []
    for ep in range(num_episodes):
        seed = seed_offset + ep
        obs, info = env.reset(seed=seed)
        agent = LowLevelLeadAimAgent()
        done = False
        while not done:
            act = agent.action(obs, info)
            data_obs.append(obs)
            data_act.append(act)
            obs, reward, terminated, truncated, info = env.step(act)
            done = terminated or truncated
        if (ep+1) % 50 == 0:
            print(f"  Episode {ep+1}/{num_episodes} done")
    env.close()
    return np.array(data_obs, dtype=np.float32), np.array(data_act, dtype=np.int64)

print("Generating dataset...")
obs_data, act_data = generate_dataset(num_episodes=300, seed_offset=1000)
print(f"Dataset size: {len(obs_data)} samples")

Generating dataset...
  Episode 50/300 done
  Episode 100/300 done
  Episode 150/300 done
  Episode 200/300 done
  Episode 250/300 done
  Episode 300/300 done
Dataset size: 511429 samples


In [ ]:
# ============================================================
# Cell 4: Train Behavioral Cloning (BC) model
# ============================================================
# Architecture: 169 -> 256 -> 128 -> 6 (pi network ของ PPO)
HIDDEN = [256, 128]
class BCPolicy(nn.Module):
    def __init__(self, obs_dim=169, act_dim=6):
        super().__init__()
        self.feature_net = nn.Sequential(
            nn.Linear(obs_dim, HIDDEN[0]), nn.ReLU(),
            nn.Linear(HIDDEN[0], HIDDEN[1]), nn.ReLU()
        )
        self.head = nn.Linear(HIDDEN[1], act_dim)

    def forward(self, x):
        feats = self.feature_net(x)
        return self.head(feats)

#data loader
dataset = TensorDataset(torch.tensor(obs_data), torch.tensor(act_data))
loader  = DataLoader(dataset, batch_size=512, shuffle=True)

bc_model = BCPolicy().to(device)
optimizer = torch.optim.Adam(bc_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

epochs = 30  
print("Training BC...")
for epoch in range(epochs):
    total_loss = 0.0
    correct = 0
    total = 0
    for batch_obs, batch_act in loader:
        batch_obs = batch_obs.to(device)
        batch_act = batch_act.to(device)
        optimizer.zero_grad()
        logits = bc_model(batch_obs)
        loss = criterion(logits, batch_act)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_obs)
        correct += (logits.argmax(dim=1) == batch_act).sum().item()
        total += len(batch_obs)
    acc = correct / total
    print(f"Epoch {epoch+1:2d} | Loss: {total_loss/total:.4f} | Acc: {acc:.3f}")


torch.save({
    'feature_net': bc_model.feature_net.state_dict(),
    'head': bc_model.head.state_dict()
}, "bc_policy_weights.pth")
print("BC weights saved to bc_policy_weights.pth")

In [ ]:
# ============================================================
# Cell: Failure Dataset from BC Policy
# ============================================================
import torch
import numpy as np
import gymnasium as gym

# ----  BC Policy  ----
class BCPolicy(torch.nn.Module):
    def __init__(self, obs_dim=169, act_dim=6):
        super().__init__()
        self.feature_net = torch.nn.Sequential(
            torch.nn.Linear(obs_dim, 256), torch.nn.ReLU(),
            torch.nn.Linear(256, 128), torch.nn.ReLU()
        )
        self.head = torch.nn.Linear(128, act_dim)

    def forward(self, x):
        return self.head(self.feature_net(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bc_model = BCPolicy().to(device)
checkpoint = torch.load("bc_policy_weights.pth", map_location=device)
bc_model.feature_net.load_state_dict(checkpoint['feature_net'])
bc_model.head.load_state_dict(checkpoint['head'])
bc_model.eval()


def calc_aim_error_bc(obs):
    best_dist = 1e9
    best_x = best_y = best_z = None
    base = 6
    for i in range(15):
        j = base + i * 8
        if obs[j + 6] > 0.5:
            x = obs[j] * 100.0
            z = obs[j + 1] * 100.0
            y = obs[j + 5] * 20.0
            dist = math.sqrt(x * x + z * z)
            if dist < best_dist:
                best_dist = dist
                best_x, best_y, best_z = x, y, z
    if best_x is None: return None
    cur_yaw = obs[3] * math.pi
    cur_pitch = obs[4] * 0.9
    ideal_yaw = math.atan2(best_z, best_x)
    xz_dist = max(math.sqrt(best_x**2 + best_z**2), 0.5)
    ideal_pitch = math.atan2(best_y - 2.3, xz_dist)
    yaw_err = abs(wrap_angle(ideal_yaw - cur_yaw))
    pitch_err = abs(ideal_pitch - cur_pitch)
    return yaw_err + pitch_err

# ---- collect failure states ----
failure_states = []
env = gym.make("Shooter-v0", render_mode=None)

for ep in range(100):  # เล่น 100 episodes
    obs, _ = env.reset()
    prev_score = 0
    while True:
        obs_tensor = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits = bc_model(obs_tensor)
            action = logits.argmax(dim=1).item()

        
        aim_err = calc_aim_error_bc(obs)  
        if action == 1 and aim_err and aim_err > 0.15:
            failure_states.append(obs.copy())

        obs, reward, terminated, truncated, info = env.step(action)

        if terminated:  # game over
            failure_states.append(obs.copy())  
            break

env.close()

failure_states = np.array(failure_states)  # list -> array
np.save("failure_states.npy", failure_states)
print(f"Saved {len(failure_states)} failure states.")